# Step 2b — Three open questions left by step 2

**Why this notebook exists.** Step 2 (`02_column_dictionary_run.ipynb`) left three questions open. Running all of step 2 again takes a long time, so this notebook holds **only the code needed for these three questions**. It reads the anime table and the user table (fast) and the 5 GB list table **once**, in chunks, keeping only a few thousand rows.

| # | Question | Why it matters |
|---|---|---|
| 1 | Do **both usernames of each shared-`user_id` pair** have rows in the list table, and are those rows the same? | If both names own rows with the same anime, one person's list would be counted twice. |
| 2 | Are the **60 early start dates** of the anime labelled `Not yet aired` real airing dates, or provisional? | It decides whether the label "Not yet aired" can be trusted for these 60 anime. |
| 3 | Are the users with an **empty profile** always the same 129 users, and do they own list rows? | We found 129 missing values in five profile columns. We assumed they are the same users. |

**How long it takes.** Parts 1 and 2 take seconds. Part 3 reads the list table in chunks of 5 million rows and takes a few minutes. Set `RUN_LISTS = False` in the first code cell to run only the fast parts.


**Where this step sits.** Step 2 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). It only measures; it changes nothing in the data. Everything is read with the safe load (only an empty cell is missing).


In [1]:
from pathlib import Path
import ast

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips the full read of the 80M-row list table
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("anime:", anime.shape, "| users:", users.shape)


anime: (14478, 31) | users: (302675, 17)


## Question 3 first (fast): the users with an empty profile

**What we found in step 2.** Five columns of the user table (`join_date`, `last_online`, `stats_mean_score`, `stats_rewatched`, `stats_episodes`) each miss exactly 129 values. We assumed these are the same 129 users, and that they are users whose profile page could not be read.

**What the cell does.** It checks that the five missing masks are identical, counts how many of these users declare anime in their list sizes (`user_watching`, ...), and checks whether the four usernames of the shared-`user_id` pairs are among them. Whether they own rows in the list table is answered in part 3.


In [2]:
miss_cols = ["join_date", "last_online", "stats_mean_score", "stats_rewatched", "stats_episodes"]
masks = {c: users[c].isna() for c in miss_cols}
print("missing values per column:", {c: int(m.sum()) for c, m in masks.items()})
print("the same users miss all five columns:", all(masks[c].equals(masks["join_date"]) for c in miss_cols))

no_profile = users[masks["join_date"]]
size_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
declared = no_profile[size_cols].sum(axis=1)
print("users with an empty profile:", len(no_profile))
print("  declare 0 anime:", int((declared == 0).sum()), "| declare more than 0:", int((declared > 0).sum()))
print("  declared list size: median", declared.median(), "| mean", round(declared.mean(), 1), "| max", declared.max())

dup_ids = users.loc[users["user_id"].duplicated(keep=False), "user_id"].unique()
pair_rows = users[users["user_id"].isin(dup_ids)].sort_values("user_id")
pair_names = set(pair_rows["username"])
print()
print("usernames of the shared-user_id pairs and whether their profile is empty:")
for _, r in pair_rows.iterrows():
    print(f"  user_id {r.user_id}: {r.username:15s} empty profile: {r.username in set(no_profile['username'])}")


missing values per column: {'join_date': 129, 'last_online': 129, 'stats_mean_score': 129, 'stats_rewatched': 129, 'stats_episodes': 129}
the same users miss all five columns: True
users with an empty profile: 129
  declare 0 anime: 2 | declare more than 0: 127
  declared list size: median 284.0 | mean 356.1 | max 2997

usernames of the shared-user_id pairs and whether their profile is empty:
  user_id 3303323: Io_Otonashi     empty profile: True
  user_id 3303323: Edengarde       empty profile: False
  user_id 4053497: Dojin_          empty profile: True
  user_id 4053497: DXJIN           empty profile: False


## The one pass over the list table (parts 1 and 2 need it)

In one pass over the 80 million rows (chunks of 5 million) the cell keeps only:
- all rows of the four usernames of the shared-`user_id` pairs (question 1);
- the rows of the anime labelled `Not yet aired`, with `my_status` and `my_score` (question 2);
- which of the empty-profile usernames own at least one row (question 3).


In [3]:
def start_date(text):
    # `aired` is the text of a Python dictionary such as {'from': '2012-01-13T00:00:00+00:00', 'to': None}
    try:
        return pd.to_datetime(ast.literal_eval(text).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

anime["start"] = anime["aired"].map(start_date)
nya = anime[anime["status"] == "Not yet aired"].copy()
bound = anime.loc[anime["status"] == "Currently Airing", "start"].max()
nya["group"] = "later or missing date"
nya.loc[nya["start"] <= bound, "group"] = "early date (on or before the bound)"
print("crawl bound (latest start date among Currently Airing):", bound)
print(nya["group"].value_counts().to_string())

if RUN_LISTS:
    nya_ids = set(nya["anime_id"])
    empty_profile_names = set(no_profile["username"])
    pair_parts, nya_parts, owners = [], [], set()
    reader = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "anime_id", "my_status", "my_score"],
        dtype={"anime_id": "int32", "my_status": "int8", "my_score": "int8"},
        chunksize=5_000_000, **na_kwargs)
    for chunk in reader:
        pair_parts.append(chunk[chunk["username"].isin(pair_names)])
        nya_parts.append(chunk.loc[chunk["anime_id"].isin(nya_ids), ["anime_id", "my_status", "my_score"]])
        owners |= set(chunk.loc[chunk["username"].isin(empty_profile_names), "username"].unique())
    pair_lists = pd.concat(pair_parts)
    nya_rows = pd.concat(nya_parts)
    print("rows kept: pairs", len(pair_lists), "| Not yet aired", len(nya_rows))


crawl bound (latest start date among Currently Airing): 2018-05-25 00:00:00+00:00
group
later or missing date                  276
early date (on or before the bound)     60
rows kept: pairs 2118 | Not yet aired 543024


## Question 1: do both usernames of a pair own list rows, and are the rows the same?

**What the cell does.** For each shared `user_id` it prints how many list rows each of the two usernames owns, how many anime appear in both lists, and for how many of those anime the status and the score are also equal. It also prints the list size the profile declares, to compare with the rows.

**How to read the result.**
- **One name owns all the rows and the other owns none:** the list is counted once. The second name is just an empty duplicate of the user row, and we can drop it.
- **Both names own rows with mostly the same anime, status and score:** one person's list is stored twice, so we must keep only one name.
- **Both own rows with different anime:** they are two different lists, and the shared id is only an id error.


In [4]:
if RUN_LISTS:
    for uid, group in pair_rows.groupby("user_id"):
        a, b = group["username"].tolist()
        ra, rb = pair_lists[pair_lists["username"] == a], pair_lists[pair_lists["username"] == b]
        declared_size = int(group[size_cols].sum(axis=1).iloc[0])
        print(f"user_id {uid}: declared list size {declared_size}")
        print(f"  {a}: {len(ra)} rows | {b}: {len(rb)} rows")
        both = ra.merge(rb, on="anime_id", suffixes=("_a", "_b"))
        same = (both["my_status_a"] == both["my_status_b"]) & (both["my_score_a"] == both["my_score_b"])
        print(f"  anime in both lists: {len(both)} | of them with the same status and score: {int(same.sum())}")


user_id 3303323: declared list size 522
  Io_Otonashi: 522 rows | Edengarde: 522 rows
  anime in both lists: 522 | of them with the same status and score: 522
user_id 4053497: declared list size 537
  Dojin_: 537 rows | DXJIN: 537 rows
  anime in both lists: 537 | of them with the same status and score: 537


## Question 2: are the early start dates real airing dates?

**The idea.** Of the 336 anime labelled `Not yet aired`, 60 have a start date on or before the crawl bound. If those dates were **real airing dates**, people would already be watching and finishing these anime, so they would look **different** from the other 276: fewer plan-to-watch rows, more watching and completed rows, more raters. If the dates are **provisional** (an expected date for an announced anime), the two groups would look **the same**.

**What the cells do.** They compare the two groups on the share of plan-to-watch rows, the number of non-plan rows per anime, the number of anime with raters, and the months of the early dates. The last cell lists the early anime with the most non-plan rows.

**How to read the result.** Similar numbers in the two groups mean the early dates are provisional. Clearly different numbers mean the label is stale for the early group.


In [5]:
if RUN_LISTS:
    rows = nya_rows.merge(nya[["anime_id", "group"]], on="anime_id")
    by_status = rows.groupby(["group", "my_status"]).size().unstack(fill_value=0)
    by_status["rows"] = by_status.sum(axis=1)
    by_status["plan_to_watch_pct"] = (by_status[6] / by_status["rows"] * 100).round(2)
    print(by_status.to_string())

    non_plan = rows[rows["my_status"] != 6].groupby("anime_id").size().rename("non_plan_rows")
    nya_rows_info = nya.merge(non_plan, on="anime_id", how="left").fillna({"non_plan_rows": 0})
    summary = nya_rows_info.groupby("group").agg(
        anime=("anime_id", "size"),
        non_plan_rows_per_anime=("non_plan_rows", "mean"),
        anime_with_raters=("scored_by", lambda s: int((s > 0).sum())),
        median_members=("members", "median"))
    print()
    print(summary.round(2).to_string())

    early = nya_rows_info[nya_rows_info["group"].str.startswith("early")]
    print()
    print("months of the early start dates:")
    print(early["start"].dt.strftime("%Y-%m").value_counts().sort_index().to_string())
    print()
    print("early anime with the most non-plan rows:")
    print(early.sort_values("non_plan_rows", ascending=False)[
        ["anime_id", "title", "start", "scored_by", "members", "non_plan_rows"]].head(10).to_string(index=False))


my_status                             1    2   3   4       6    rows  plan_to_watch_pct
group                                                                                  
early date (on or before the bound)  64  864  22  12   52597   53559              98.20
later or missing date                17    2  20   4  489422  489465              99.99

                                     anime  non_plan_rows_per_anime  anime_with_raters  median_members
group                                                                                                 
early date (on or before the bound)     60                    16.03                  2          1205.0
later or missing date                  276                     0.16                 44          2172.0

months of the early start dates:
start
2018-01     1
2018-04     3
2018-05    56

early anime with the most non-plan rows:
 anime_id                                                   title                     start  scored_by  members

## Question 3 (second half): do the empty-profile users own list rows?


In [6]:
if RUN_LISTS:
    print("users with an empty profile:", len(empty_profile_names))
    print("  of them with at least one list row:", len(owners))
    print("  of them without any list row:", len(empty_profile_names - owners))
    declared_by_name = no_profile.set_index("username")[size_cols].sum(axis=1)
    no_rows = declared_by_name[list(empty_profile_names - owners)]
    print("  of those without rows, declaring more than 0 anime:", int((no_rows > 0).sum()))


users with an empty profile: 129
  of them with at least one list row: 127
  of them without any list row: 2
  of those without rows, declaring more than 0 anime: 0


## Observations

**TODO (fill in after the run):**
1. Question 3: are the five missing masks the same? How many of the 129 declare anime, and how many own rows? Are the two empty-profile names of the pairs among them?
2. Question 1: how many rows does each name own, and do the lists overlap?
3. Question 2: are the early group and the later group similar in the share of plan-to-watch rows, in the non-plan rows per anime and in raters? Are the early dates concentrated in a few months?
